# 12 — Leave-One-Event-Out (LOEO) Validation

For each known market-driven depeg event in the test period, retrain CatBoost using the
tuned hyperparameters from NB10 on all data **excluding a window around that event**.
Then predict on the held-out window and check whether the model still fires an alert
before the depeg onset.

This tests whether the model generalises across *events*, not just across time.
No re-tuning — uses NB10's best_params directly.


In [1]:
import os, sys, warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import json
from catboost import CatBoostClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/Capstone')
else:
    _candidates = [Path.cwd()] + list(Path.cwd().parents)
    ROOT = next((p for p in _candidates if (p / 'config' / 'settings.py').exists()), None)
    if ROOT is None:
        raise FileNotFoundError('Could not find project root')

MODELS_DIR   = ROOT / 'data' / 'models'
FEATURES_DIR = ROOT / 'data' / 'processed' / 'features'
print(f'ROOT: {ROOT}')


ROOT: /Users/robertspringett/Education/CMU_MSBA/capstone_5min_global


## 1. Load Data & NB10 Metadata


In [2]:
with open(MODELS_DIR / 'downside_depeg_meta.json') as f:
    meta = json.load(f)

TARGET      = meta['target']
best_params = meta['best_params']
feat_full   = meta['feat_full']
TRAIN_END   = pd.Timestamp(meta['train_end'])
VAL_START   = pd.Timestamp('2021-01-01', tz='UTC')
VAL_END     = pd.Timestamp('2022-01-01', tz='UTC')

df = pd.read_parquet(FEATURES_DIR / 'pooled_5m.parquet')
df.index = pd.to_datetime(df.index, utc=True)
df_stable = df[df['depeg'] == 0].dropna(subset=[TARGET])
feat_full  = [f for f in feat_full if f in df_stable.columns]

print(f'Target    : {TARGET}')
print(f'Features  : {len(feat_full)}')
print(f'Algorithm : CatBoost  iterations={best_params["iterations"]}  depth={best_params["depth"]}')
print(f'Total bars: {len(df_stable):,}')
print(f'Val period for threshold calibration: {VAL_START.date()} → {VAL_END.date()}')


Target    : depeg_next_4h_down
Features  : 40
Algorithm : CatBoost  iterations=200  depth=4
Total bars: 3,099,663
Val period for threshold calibration: 2021-01-01 → 2022-01-01


## 2. Define Events

Five market-driven depeg events in the test period (UST and BUSD excluded — non-market-driven).
Each event has a holdout window: 14 days before the first onset through 3 days after.
The model is retrained on everything outside that window, then evaluated inside it.


In [3]:
# Each entry: (name, coins_affected, first_onset_utc, holdout_start, holdout_end)
# holdout window = 14 days pre-onset to 3 days post-onset
def event_window(onset_str, pre_days=14, post_days=3):
    onset = pd.Timestamp(onset_str, tz='UTC')
    return onset - pd.Timedelta(days=pre_days), onset + pd.Timedelta(days=post_days)

EVENTS = [
    {
        'name':   'FTX Collapse',
        'date':   '2022-11-10',
        'coins':  ['usdt'],
        'onset':  '2022-11-10 11:10:00',
    },
    {
        'name':   'SVB Bank Run',
        'date':   '2023-03-11',
        'coins':  ['usdc', 'dai', 'usdt'],
        'onset':  '2023-03-11 01:00:00',  # USDC first mover
    },
    {
        'name':   'USDe Jun 2024',
        'date':   '2024-06-02',
        'coins':  ['usde'],
        'onset':  '2024-06-02 22:15:00',
    },
    {
        'name':   'USDe Jul 2024',
        'date':   '2024-07-22',
        'coins':  ['usde'],
        'onset':  '2024-07-22 19:55:00',
    },
    {
        'name':   'USDe Feb 2025',
        'date':   '2025-02-21',
        'coins':  ['usde'],
        'onset':  '2025-02-21 15:40:00',
    },
    {
        'name':   'Oct 2025 Stress',
        'date':   '2025-10-10',
        'coins':  ['usdt', 'usde'],
        'onset':  '2025-10-10 22:40:00',
    },
]

for e in EVENTS:
    e['onset_ts'] = pd.Timestamp(e['onset'], tz='UTC')
    e['holdout_start'], e['holdout_end'] = event_window(e['onset'])
    print(f'{e["name"]:<20}  onset={e["onset"][:10]}  '
          f'holdout={str(e["holdout_start"])[:10]} → {str(e["holdout_end"])[:10]}  '
          f'coins={e["coins"]}')


FTX Collapse          onset=2022-11-10  holdout=2022-10-27 → 2022-11-13  coins=['usdt']
SVB Bank Run          onset=2023-03-11  holdout=2023-02-25 → 2023-03-14  coins=['usdc', 'dai', 'usdt']
USDe Jun 2024         onset=2024-06-02  holdout=2024-05-19 → 2024-06-05  coins=['usde']
USDe Jul 2024         onset=2024-07-22  holdout=2024-07-08 → 2024-07-25  coins=['usde']
USDe Feb 2025         onset=2025-02-21  holdout=2025-02-07 → 2025-02-24  coins=['usde']
Oct 2025 Stress       onset=2025-10-10  holdout=2025-09-26 → 2025-10-13  coins=['usdt', 'usde']


## 3. LOEO Training & Evaluation

For each event:
1. Remove holdout window from training data
2. Retrain CatBoost with NB10 best_params
3. Predict on held-out bars for affected coins
4. Find first alert within 24h before onset
5. Report lead time


In [4]:
LOOKBACK  = pd.Timedelta('24h')
PCT_SWEEP = [0.1, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0]
results   = []

for e in EVENTS:
    name    = e['name']
    onset   = e['onset_ts']
    h_start = e['holdout_start']
    h_end   = e['holdout_end']
    coins   = e['coins']

    # ── Split ─────────────────────────────────────────────────────
    in_holdout = (df_stable.index >= h_start) & (df_stable.index <= h_end)
    in_val     = (df_stable.index >= VAL_START) & (df_stable.index < VAL_END)

    train_df   = df_stable[~in_holdout & ~in_val & (df_stable.index < TRAIN_END)]
    val_df     = df_stable[in_val & ~in_holdout]
    holdout_df = df_stable[in_holdout & df_stable['coin'].isin(coins)]

    y_tr  = train_df[TARGET].values.astype(float)
    y_val = val_df[TARGET].values.astype(float)
    y_ho  = holdout_df[TARGET].values.astype(float)

    if y_tr.sum() < 10 or y_val.sum() < 3 or len(holdout_df) == 0:
        print(f'{name}: insufficient data — skipping')
        continue

    imp   = SimpleImputer(strategy='constant', fill_value=0.0)
    X_tr  = imp.fit_transform(train_df[feat_full].values)
    X_val = imp.transform(val_df[feat_full].values)
    X_ho  = imp.transform(holdout_df[feat_full].values)

    # ── Train ─────────────────────────────────────────────────────
    model = CatBoostClassifier(**{**best_params,
                                  'auto_class_weights': 'Balanced',
                                  'random_seed': 42, 'verbose': 0})
    model.fit(X_tr, y_tr)

    # ── Calibrate threshold on this model's val predictions ───────
    p_val_this = model.predict_proba(X_val)[:, 1]
    best_f2_val, best_pct, threshold = 0, None, None
    for pct in PCT_SWEEP:
        t  = np.percentile(p_val_this, 100 - pct)
        al = p_val_this >= t
        if al.sum() == 0: continue
        tp   = int((al & (y_val == 1)).sum())
        prec = tp / al.sum()
        rec  = tp / int(y_val.sum())
        f2   = (5*prec*rec)/(4*prec+rec) if (prec+rec) > 0 else 0
        if f2 > best_f2_val:
            best_f2_val, best_pct, threshold = f2, pct, t

    # ── Predict on holdout ────────────────────────────────────────
    p_ho     = model.predict_proba(X_ho)[:, 1]
    al       = p_ho >= threshold
    alerts_s = pd.Series(al.astype(bool), index=holdout_df.index)

    base   = float(y_ho.mean()) if y_ho.mean() > 0 else float('nan')
    pr_auc = average_precision_score(y_ho, p_ho) if y_ho.sum() > 0 else float('nan')
    lift   = pr_auc / base if base > 0 else float('nan')

    # F2 on holdout at calibrated threshold
    if al.sum() > 0 and y_ho.sum() > 0:
        tp_ho   = int((al & (y_ho == 1)).sum())
        prec_ho = tp_ho / al.sum()
        rec_ho  = tp_ho / int(y_ho.sum())
        f2_ho   = (5*prec_ho*rec_ho)/(4*prec_ho+rec_ho) if (prec_ho+rec_ho) > 0 else 0
    else:
        prec_ho = rec_ho = f2_ho = float('nan')

    # ── Lead time ─────────────────────────────────────────────────
    pre_alerts = alerts_s[(alerts_s.index >= onset - LOOKBACK) & (alerts_s.index < onset)]
    fired      = pre_alerts[pre_alerts]
    if len(fired) > 0:
        first_alert = fired.index[0]
        lead_m      = (onset - first_alert).total_seconds() / 60
        detected    = True
    else:
        first_alert = None
        lead_m      = None
        detected    = False

    results.append(dict(
        name=name, onset=onset, coins=','.join(coins),
        detected=detected, lead_min=lead_m,
        pr_auc=pr_auc, lift=lift, f2=f2_ho,
        recall=rec_ho, precision=prec_ho,
        threshold_pct=best_pct,
        n_train=len(train_df), n_holdout=len(holdout_df),
        n_pos_holdout=int(y_ho.sum()),
    ))

    status = f'DETECTED  lead={lead_m:.0f} min ({lead_m/60:.1f} h)' if detected else 'NOT DETECTED'
    f2_str = f'{f2_ho:.4f}' if f2_ho == f2_ho else 'n/a'
    print(f'{name:<22}  {status}  |  threshold=top {best_pct}%  PR-AUC={pr_auc:.4f}  F2={f2_str}  lift={lift:.1f}x')


FTX Collapse            DETECTED  lead=390 min (6.5 h)  |  threshold=top 0.5%  PR-AUC=0.2555  F2=0.3534  lift=25.9x


SVB Bank Run            NOT DETECTED  |  threshold=top 0.5%  PR-AUC=0.3498  F2=0.2191  lift=22.1x


USDe Jun 2024           NOT DETECTED  |  threshold=top 0.5%  PR-AUC=nan  F2=n/a  lift=nanx


USDe Jul 2024           DETECTED  lead=20 min (0.3 h)  |  threshold=top 0.5%  PR-AUC=0.5408  F2=0.2889  lift=24.8x


USDe Feb 2025           DETECTED  lead=5 min (0.1 h)  |  threshold=top 0.5%  PR-AUC=0.3933  F2=0.2261  lift=20.3x


Oct 2025 Stress         DETECTED  lead=10 min (0.2 h)  |  threshold=top 0.5%  PR-AUC=0.0495  F2=0.0503  lift=10.1x


## 4. Summary


In [5]:
print('=' * 80)
print('  LOEO Validation Summary')
print('=' * 80)
print(f'  {"Event":<22}  {"Coins":<16}  {"Det":>4}  {"Lead time":>14}  {"PR-AUC":>7}  {"F2":>7}  {"Lift":>7}')
print(f'  {"-"*22}  {"-"*16}  {"-"*4}  {"-"*14}  {"-"*7}  {"-"*7}  {"-"*7}')

detected_count = 0
lead_times, pr_aucs, f2s, lifts = [], [], [], []
for r in results:
    lead_str   = f'{r["lead_min"]:.0f}m ({r["lead_min"]/60:.1f}h)' if r["lead_min"] else '---'
    pr_str     = f'{r["pr_auc"]:.4f}' if r["pr_auc"] == r["pr_auc"] else 'n/a'
    f2_str     = f'{r["f2"]:.4f}'     if r["f2"]     == r["f2"]     else 'n/a'
    lift_str   = f'{r["lift"]:.1f}x'  if r["lift"]   == r["lift"]   else 'n/a'
    mark       = 'YES' if r['detected'] else 'NO'
    print(f'  {r["name"]:<22}  {r["coins"]:<16}  {mark:>4}  {lead_str:>14}  {pr_str:>7}  {f2_str:>7}  {lift_str:>7}')
    if r['detected']:
        detected_count += 1
        lead_times.append(r['lead_min'])
    if r['pr_auc'] == r['pr_auc']:
        pr_aucs.append(r['pr_auc'])
        lifts.append(r['lift'])
    if r['f2'] == r['f2']:
        f2s.append(r['f2'])

print()
print(f'  Events detected  : {detected_count} / {len(results)}  ({100*detected_count/len(results):.0f}%)')
if lead_times:
    print(f'  Median lead time : {np.median(lead_times):.0f} min  ({np.median(lead_times)/60:.1f} h)')
    print(f'  Min / Max        : {min(lead_times):.0f} / {max(lead_times):.0f} min')
if pr_aucs:
    print(f'  Mean PR-AUC      : {np.mean(pr_aucs):.4f}')
    print(f'  Mean F2          : {np.mean(f2s):.4f}')
    print(f'  Mean Lift        : {np.mean(lifts):.1f}x')
print('=' * 80)


  LOEO Validation Summary
  Event                   Coins              Det       Lead time   PR-AUC       F2     Lift
  ----------------------  ----------------  ----  --------------  -------  -------  -------
  FTX Collapse            usdt               YES     390m (6.5h)   0.2555   0.3534    25.9x
  SVB Bank Run            usdc,dai,usdt       NO             ---   0.3498   0.2191    22.1x
  USDe Jun 2024           usde                NO             ---      n/a      n/a      n/a
  USDe Jul 2024           usde               YES      20m (0.3h)   0.5408   0.2889    24.8x
  USDe Feb 2025           usde               YES       5m (0.1h)   0.3933   0.2261    20.3x
  Oct 2025 Stress         usdt,usde          YES      10m (0.2h)   0.0495   0.0503    10.1x

  Events detected  : 4 / 6  (67%)
  Median lead time : 15 min  (0.2 h)
  Min / Max        : 5 / 390 min
  Mean PR-AUC      : 0.3178
  Mean F2          : 0.2275
  Mean Lift        : 20.6x
